# XY FSS — O(2) Universality Validation

Finite-size scaling analysis of the classical XY model on cubic lattices.
Target: recover O(2) exponents ν=0.6717, γ=1.3177, β=0.3486 and Tc=2.2016 J/kB.

## Sections
1. Load `xy_fss_N*.csv` — plot E, |M|, Cv, χ vs T with error bars
2. Binder cumulant U(T) crossings → Tc with uncertainty
3. Peak scaling: χ_max ~ L^{γ/ν}, M(Tc) ~ L^{-β/ν}
4. Scaling collapse → ν
5. Summary table: measured vs O(2) theory

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path('data')
SIZES = [8, 12, 16, 20, 24, 32]

dfs = {}
for n in SIZES:
    path = DATA_DIR / f'xy_fss_N{n}.csv'
    if path.exists():
        dfs[n] = pd.read_csv(path)
    else:
        print(f'Missing: {path}')

print(f'Loaded {len(dfs)} size(s):', list(dfs.keys()))
if dfs:
    display(next(iter(dfs.values())).head())

In [ ]:
## 1. Raw observables — E, |M|, Cv, χ vs T with error bars

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
colors = plt.cm.viridis(np.linspace(0, 1, len(SIZES)))

for ax, (col, label) in zip(axes.flat, [
    ('E',   'Energy per spin  E/N'),
    ('M',   'Magnetisation |m|'),
    ('Cv',  'Heat capacity Cv'),
    ('chi', 'Susceptibility χ'),
]):
    err_col = col + '_err'
    for (n, df), c in zip(dfs.items(), colors):
        ax.errorbar(df['T'], df[col], yerr=df[err_col],
                    label=f'L={n}', color=c, capsize=2, linewidth=1.2)
    ax.set_xlabel('T  (J/kB)')
    ax.set_ylabel(label)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Classical XY model — cubic lattice', fontsize=13)
plt.tight_layout()
plt.savefig('figures/xy_observables.png', dpi=150)
plt.show()
print('Saved figures/xy_observables.png')

In [ ]:
## 2. Binder cumulant crossings → Tc

def binder(df):
    """U = 1 - <m4> / (3 <m2>^2)"""
    return 1.0 - df['M4'] / (3.0 * df['M2']**2)

# Plot U(T) per size
fig, ax = plt.subplots(figsize=(7, 5))
colors = plt.cm.viridis(np.linspace(0, 1, len(dfs)))
for (n, df), c in zip(dfs.items(), colors):
    u = binder(df)
    ax.plot(df['T'], u, label=f'L={n}', color=c, linewidth=1.5)
ax.axvline(2.2016, color='k', linestyle='--', alpha=0.5, label='Tc theory=2.2016')
ax.set_xlabel('T  (J/kB)')
ax.set_ylabel('Binder cumulant U')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
ax.set_title('Binder cumulant — crossings give Tc')
plt.tight_layout()
plt.savefig('figures/xy_binder.png', dpi=150)
plt.show()

# Find pairwise crossings by linear interpolation
from itertools import combinations

def crossing_temp(df1, df2):
    """Linear-interpolation crossing of two Binder curves."""
    u1, u2 = binder(df1).values, binder(df2).values
    t = df1['T'].values
    diff = u1 - u2
    # find sign change
    for i in range(len(diff) - 1):
        if diff[i] * diff[i+1] < 0:
            # linear interpolation
            tc = t[i] - diff[i] * (t[i+1] - t[i]) / (diff[i+1] - diff[i])
            return tc
    return np.nan

size_list = sorted(dfs.keys())
crossings = []
for na, nb in combinations(size_list, 2):
    tc = crossing_temp(dfs[na], dfs[nb])
    crossings.append(tc)
    print(f'  L={na} × L={nb}: Tc = {tc:.4f}')

crossings = np.array([c for c in crossings if not np.isnan(c)])
Tc_mean = np.mean(crossings)
Tc_err  = np.std(crossings)
print(f'\nTc = {Tc_mean:.4f} ± {Tc_err:.4f}  (theory: 2.2016)')

In [ ]:
## 3. Peak scaling: χ_max ~ L^{γ/ν},  M(Tc) ~ L^{-β/ν}

from scipy.optimize import curve_fit

Tc = Tc_mean  # use measured Tc from Binder crossings

Ls, chi_max, m_at_tc = [], [], []
for n, df in sorted(dfs.items()):
    Ls.append(n)
    chi_max.append(df['chi'].max())
    # M at the temperature closest to Tc
    idx = (df['T'] - Tc).abs().idxmin()
    m_at_tc.append(df.loc[idx, 'M'])

Ls       = np.array(Ls, dtype=float)
chi_max  = np.array(chi_max)
m_at_tc  = np.array(m_at_tc)

def powerlaw(L, a, exp):
    return a * L**exp

popt_chi, pcov_chi = curve_fit(powerlaw, Ls, chi_max, p0=[1.0, 1.96])
popt_m,   pcov_m   = curve_fit(powerlaw, Ls, m_at_tc,  p0=[1.0, -0.52])

gamma_over_nu = popt_chi[1]
beta_over_nu  = -popt_m[1]
gamma_over_nu_err = np.sqrt(pcov_chi[1,1])
beta_over_nu_err  = np.sqrt(pcov_m[1,1])

print(f'γ/ν = {gamma_over_nu:.4f} ± {gamma_over_nu_err:.4f}  (O(2) theory: {1.3177/0.6717:.4f})')
print(f'β/ν = {beta_over_nu:.4f}  ± {beta_over_nu_err:.4f}   (O(2) theory: {0.3486/0.6717:.4f})')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
L_fine = np.linspace(Ls.min(), Ls.max(), 100)

ax = axes[0]
ax.loglog(Ls, chi_max, 'o', label='data')
ax.loglog(L_fine, powerlaw(L_fine, *popt_chi), '--',
          label=f'fit: γ/ν={gamma_over_nu:.3f}')
ax.set_xlabel('L'); ax.set_ylabel('χ_max'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Susceptibility peak scaling')

ax = axes[1]
ax.loglog(Ls, m_at_tc, 'o', label='data')
ax.loglog(L_fine, powerlaw(L_fine, *popt_m), '--',
          label=f'fit: β/ν={beta_over_nu:.3f}')
ax.set_xlabel('L'); ax.set_ylabel('M(Tc)'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Magnetisation at Tc scaling')

plt.tight_layout()
plt.savefig('figures/xy_peak_scaling.png', dpi=150)
plt.show()

In [ ]:
## 4. Scaling collapse → ν
#
# Collapse: M · L^{β/ν}  vs  (T - Tc) · L^{1/ν}
# Use O(2) theory exponents first, then optimise ν by minimising spread.

from scipy.optimize import minimize_scalar

NU_THEORY  = 0.6717
BETA_THEORY = 0.3486

def collapse_spread(nu, dfs=dfs, Tc=Tc, beta_over_nu=BETA_THEORY/NU_THEORY):
    """RMS spread of scaled curves — minimise over ν."""
    xs, ys = [], []
    for n, df in dfs.items():
        x = (df['T'].values - Tc) * n**(1.0/nu)
        y = df['M'].values * n**beta_over_nu
        xs.append(x); ys.append(y)
    # Interpolate all onto a common x grid and measure variance
    x_common = np.linspace(-5, 5, 200)
    curves = []
    for x, y in zip(xs, ys):
        idx = np.argsort(x)
        yy = np.interp(x_common, x[idx], y[idx],
                       left=np.nan, right=np.nan)
        curves.append(yy)
    stack = np.array(curves)
    spread = np.nanstd(stack, axis=0)
    return np.nanmean(spread)

result = minimize_scalar(collapse_spread, bounds=(0.4, 1.0), method='bounded')
nu_fit = result.x
print(f'ν_fit = {nu_fit:.4f}  (O(2) theory: {NU_THEORY})')

# Plot collapse with fitted ν
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = plt.cm.viridis(np.linspace(0, 1, len(dfs)))

for (title, nu), ax in zip([('Theory ν=0.6717', NU_THEORY), (f'Fitted ν={nu_fit:.4f}', nu_fit)], axes):
    for (n, df), c in zip(sorted(dfs.items()), colors):
        x = (df['T'].values - Tc) * n**(1.0/nu)
        y = df['M'].values * n**(BETA_THEORY/nu)
        ax.plot(x, y, label=f'L={n}', color=c, linewidth=1.2)
    ax.set_xlim(-6, 6); ax.set_ylim(0, None)
    ax.set_xlabel('(T − Tc) · L^{1/ν}')
    ax.set_ylabel('M · L^{β/ν}')
    ax.set_title(title); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('figures/xy_collapse.png', dpi=150)
plt.show()

In [ ]:
## 5. Summary table: measured vs O(2) theory

# Derive γ and β from the fitted ratios and ν
gamma_fit = gamma_over_nu * nu_fit
beta_fit  = beta_over_nu  * nu_fit
gamma_err = gamma_over_nu_err * nu_fit
beta_err  = beta_over_nu_err  * nu_fit

print('=' * 55)
print(f'{"Quantity":<12} {"Measured":>14} {"O(2) theory":>14}')
print('-' * 55)
print(f'{"Tc (J/kB)":<12} {Tc_mean:>10.4f}±{Tc_err:.4f} {"2.2016":>14}')
print(f'{"ν":<12} {nu_fit:>14.4f} {NU_THEORY:>14.4f}')
print(f'{"γ/ν":<12} {gamma_over_nu:>10.4f}±{gamma_over_nu_err:.4f} {1.3177/0.6717:>14.4f}')
print(f'{"β/ν":<12} {beta_over_nu:>10.4f}±{beta_over_nu_err:.4f}  {0.3486/0.6717:>14.4f}')
print(f'{"γ":<12} {gamma_fit:>10.4f}±{gamma_err:.4f} {1.3177:>14.4f}')
print(f'{"β":<12} {beta_fit:>10.4f}±{beta_err:.4f}  {0.3486:>14.4f}')
print('=' * 55)

# Percentage errors
print(f'\nTc error:  {100*abs(Tc_mean-2.2016)/2.2016:.2f}%')
print(f'ν  error:  {100*abs(nu_fit-NU_THEORY)/NU_THEORY:.2f}%')
print(f'γ  error:  {100*abs(gamma_fit-1.3177)/1.3177:.2f}%')
print(f'β  error:  {100*abs(beta_fit-0.3486)/0.3486:.2f}%')